# Colab bootstrap

Run the cell below once after connecting to a new Colab runtime. It mounts Google Drive, clones or updates the repository, installs the project, and configures the paths used by the detection YAML.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/terriljoel/retrieval-grounded-remote-sensing.git"
REPO_REF = "feat/object-detection-YOLO"  # Change to main after merging.
REPO_DIR = Path("/content/retrieval-grounded-remote-sensing")
SHARED_ROOT = Path("/content/drive/Othercomputers/My laptop/shared_resources")

drive.mount("/content/drive")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Repository path exists but is not a Git clone: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)],
    check=True,
)

if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared resources were not found at {SHARED_ROOT}. "
        "Check the Google Drive computer and folder names."
    )

os.environ.update({
    "SHARED_RESOURCES_ROOT": str(SHARED_ROOT),
    "RAW_DATASET_ROOT": "/content/datasets/raw",
    "PROCESSED_DATASET_ROOT": "/content/datasets/processed",
    "MANIFEST_ROOT": str(SHARED_ROOT / "datasets" / "manifests"),
    "EXPERIMENT_OUTPUT_ROOT": str(SHARED_ROOT / "experiment_outputs"),
    "JOB_LOG_ROOT": str(SHARED_ROOT / "experiment_outputs" / "job_logs"),
})

Path(os.environ["MANIFEST_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["EXPERIMENT_OUTPUT_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["JOB_LOG_ROOT"]).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

print(f"Ready. Working directory: {Path.cwd()}")
print("Next command:")
print("!rs-prepare-detector --config configs/detection/yolov8n_baseline.yaml")